In [1]:
import numpy as np
import pandas as pd

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from google.colab import files
uploaded = files.upload()

Saving claims_phase1.parquet to claims_phase1.parquet
Saving members_phase1.parquet to members_phase1.parquet
Saving providers_phase1.parquet to providers_phase1.parquet


In [4]:
claims = pd.read_parquet ("claims_phase1.parquet")

In [5]:
members = pd.read_parquet("members_phase1.parquet")

In [6]:
providers = pd.read_parquet("providers_phase1.parquet")

In [7]:
providers.sample(10)

,provider_id,speciality,provider_region
889,P889,Specialist,West
751,P751,Specialist,North
176,P176,Specialist,West
455,P455,PrimaryCare,West
669,P669,Specialist,Central
543,P543,PrimaryCare,Costal
961,P961,Specialist,Costal
556,P556,Specialist,West
45,P45,Specialist,South
989,P989,PrimaryCare,Costal


In [8]:
members.sample(10)

,member_id,age,gender,region,chronic_count,behavioral_flag,risk_level,annual_claims,pcp_engaged
12443,M12443,32,M,South,0,0,Medium,14,0
46700,M46700,41,F,South,1,1,Medium,14,0
3331,M3331,23,M,West,1,0,Medium,15,1
45132,M45132,42,F,East,0,0,Low,6,1
2215,M2215,36,F,North,0,1,Low,7,0
35407,M35407,59,F,North,1,0,Low,8,1
16484,M16484,27,F,East,0,0,Low,7,1
12152,M12152,63,F,Central,0,0,Low,8,1
8094,M8094,51,F,Central,1,0,Low,8,1
25867,M25867,63,M,South,1,0,Low,7,1


In [9]:
claims.sample(2)

,member_id,age,gender,region,chronic_count,behavioral_flag,risk_level,annual_claims,claim_id,date,place_of_service,diagnosis_group,admission_flag,allowed_amount,avoidable_er,provider_id,speciality,provider_region,pcp_engaged,region_multiplier
238020,M22770,48,F,South,0,0,Low,7,C238020,2024-10-15,OP,low_acuity,0,72.768854,0,P492,Specialist,Costal,0,1.00
70958,M6766,35,M,Costal,1,1,Medium,14,C70958,2024-06-10,OP,chronic,0,189.490630,0,P669,Specialist,Central,0,1.08


In [10]:
claims ["allowed_amount"] = claims ["allowed_amount"].fillna(0)

In [11]:
member_agg = claims.groupby ("member_id").agg (
    total_cost = ("allowed_amount" ,"sum"),
    avg_cost_per_claim =("allowed_amount", "mean"),
    total_claims = ("claim_id", "count"),
    avoidable_er_count = ("avoidable_er", "sum"),
    ip_admissions = ("admission_flag", "sum")
).reset_index()

In [12]:
member_agg.sample(2)

,member_id,total_cost,avg_cost_per_claim,total_claims,avoidable_er_count,ip_admissions
28234,M35408,44865.661493,5608.207687,8,0,3
11054,M19947,34752.368598,4344.046075,8,0,1


In [13]:
member_agg.shape

(50000, 6)

In [14]:
member_agg.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   member_id           50000 non-null  object 
 1   total_cost          50000 non-null  float64
 2   avg_cost_per_claim  50000 non-null  float64
 3   total_claims        50000 non-null  int64  
 4   avoidable_er_count  50000 non-null  int64  
 5   ip_admissions       50000 non-null  int64  
dtypes: float64(2), int64(3), object(1)
memory usage: 2.3+ MB


In [15]:
er_counts= (
    claims.assign (er_flag =claims["place_of_service"]=="ER")
    .groupby("member_id")["er_flag"]
    .sum()
    .reset_index(name= "er_visits")
            )


In [16]:
member_agg = member_agg.merge(er_counts, on= "member_id", how= "left")

In [17]:
member_agg["er_visits"]= member_agg["er_visits"].fillna(0).astype(int)

In [18]:
member_agg.sample(2)

,member_id,total_cost,avg_cost_per_claim,total_claims,avoidable_er_count,ip_admissions,er_visits
214,M1019,285211.671615,12400.507462,23,2,4,4
14193,M22771,3122.934133,446.133448,7,0,0,1


In [19]:
member_agg.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   member_id           50000 non-null  object 
 1   total_cost          50000 non-null  float64
 2   avg_cost_per_claim  50000 non-null  float64
 3   total_claims        50000 non-null  int64  
 4   avoidable_er_count  50000 non-null  int64  
 5   ip_admissions       50000 non-null  int64  
 6   er_visits           50000 non-null  int64  
dtypes: float64(2), int64(4), object(1)
memory usage: 2.7+ MB


In [20]:
pcp_counts= (
    claims.assign (pcp_flag = claims["speciality"]=="PrimaryCare")
    .groupby("member_id")["pcp_flag"]
    .sum()
    .reset_index(name = "pcp_visits")
)

In [21]:
member_agg = member_agg.merge(pcp_counts, on= "member_id", how= "left")

In [22]:
member_agg["pcp_visits"]= member_agg["pcp_visits"].fillna(0).astype(int)

In [23]:
member_agg.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   member_id           50000 non-null  object 
 1   total_cost          50000 non-null  float64
 2   avg_cost_per_claim  50000 non-null  float64
 3   total_claims        50000 non-null  int64  
 4   avoidable_er_count  50000 non-null  int64  
 5   ip_admissions       50000 non-null  int64  
 6   er_visits           50000 non-null  int64  
 7   pcp_visits          50000 non-null  int64  
dtypes: float64(2), int64(5), object(1)
memory usage: 3.1+ MB


In [24]:
model_df = member_agg.merge(
    members,
    on = "member_id",
    how= "left"
)

In [25]:
model_df.shape

(50000, 16)

In [26]:
model_df.isna().sum()

,0
member_id,0
total_cost,0
avg_cost_per_claim,0
total_claims,0
avoidable_er_count,0
ip_admissions,0
er_visits,0
pcp_visits,0
age,0
gender,0


In [27]:
model_df.sample(5)

,member_id,total_cost,avg_cost_per_claim,total_claims,avoidable_er_count,ip_admissions,er_visits,pcp_visits,age,gender,region,chronic_count,behavioral_flag,risk_level,annual_claims,pcp_engaged
25498,M32946,45138.611679,3224.186548,14,1,2,4,1,60,M,East,1,0,Medium,14,0
22998,M30696,17486.807451,2914.467909,6,0,1,0,4,6,F,North,0,0,Low,6,0
6488,M15837,20875.112876,1491.079491,14,0,1,0,6,47,M,North,1,0,Medium,14,1
28881,M35991,89777.594431,6412.685317,14,2,4,4,4,0,M,Costal,1,0,Medium,14,0
2443,M12196,1456.860288,208.122898,7,0,0,0,3,52,F,North,1,0,Low,7,1


In [28]:
model_df["member_id"].nunique()

50000

In [29]:
model_df["has_avoidable_er"]= np.where (
    model_df["avoidable_er_count"]>0,
    1,
    0
)


In [30]:
model_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 17 columns):
 #   Column              Non-Null Count  Dtype   
---  ------              --------------  -----   
 0   member_id           50000 non-null  object  
 1   total_cost          50000 non-null  float64 
 2   avg_cost_per_claim  50000 non-null  float64 
 3   total_claims        50000 non-null  int64   
 4   avoidable_er_count  50000 non-null  int64   
 5   ip_admissions       50000 non-null  int64   
 6   er_visits           50000 non-null  int64   
 7   pcp_visits          50000 non-null  int64   
 8   age                 50000 non-null  int64   
 9   gender              50000 non-null  object  
 10  region              50000 non-null  object  
 11  chronic_count       50000 non-null  int64   
 12  behavioral_flag     50000 non-null  int64   
 13  risk_level          50000 non-null  category
 14  annual_claims       50000 non-null  int64   
 15  pcp_engaged         50000 non-null  

In [31]:
model_df.shape

(50000, 17)

In [32]:
model_df["er_rate"]= np.where(
    model_df["total_claims"]>0,
    model_df["er_visits"]/ model_df["total_claims"],
    0 )

In [33]:
model_df["er_rate"] = model_df["er_rate"].fillna(0)

In [34]:
model_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype   
---  ------              --------------  -----   
 0   member_id           50000 non-null  object  
 1   total_cost          50000 non-null  float64 
 2   avg_cost_per_claim  50000 non-null  float64 
 3   total_claims        50000 non-null  int64   
 4   avoidable_er_count  50000 non-null  int64   
 5   ip_admissions       50000 non-null  int64   
 6   er_visits           50000 non-null  int64   
 7   pcp_visits          50000 non-null  int64   
 8   age                 50000 non-null  int64   
 9   gender              50000 non-null  object  
 10  region              50000 non-null  object  
 11  chronic_count       50000 non-null  int64   
 12  behavioral_flag     50000 non-null  int64   
 13  risk_level          50000 non-null  category
 14  annual_claims       50000 non-null  int64   
 15  pcp_engaged         50000 non-null  

In [35]:
model_df.shape

(50000, 18)

In [36]:
model_df.sample(5)

,member_id,total_cost,avg_cost_per_claim,total_claims,avoidable_er_count,ip_admissions,er_visits,pcp_visits,age,gender,region,chronic_count,behavioral_flag,risk_level,annual_claims,pcp_engaged,has_avoidable_er,er_rate
6383,M15742,2027.805936,337.967656,6,1,0,1,3,37,M,Central,0,0,Low,6,1,1,0.166667
31421,M38277,3372.285513,562.047585,6,0,0,2,3,37,M,South,1,0,Low,6,0,0,0.333333
36039,M42432,76625.479877,3831.273994,20,1,1,3,7,19,F,Costal,1,0,High,20,1,1,0.150000
22730,M30454,2695.891698,449.315283,6,1,0,1,2,30,M,West,0,0,Low,6,0,1,0.166667
33275,M39946,50635.822516,3616.844465,14,0,2,0,4,45,M,West,0,0,Medium,14,1,0,0.000000


In [37]:
members["chronic_count"]= (members["chronic_count"].clip(upper=4)).astype(int)

In [38]:
members["chronic_count"].describe()

,chronic_count
count,50000.000000
mean,0.527300
std,0.789169
min,0.000000
25%,0.000000
50%,0.000000
75%,1.000000
max,4.000000


In [39]:
hcc_cols= ["diabetes_flag", "chf_flag", "copd_flag", "renal_flag"]

In [40]:
for cols in hcc_cols:
  members[cols]=0

In [41]:
members.sample(5)

,member_id,age,gender,region,chronic_count,behavioral_flag,risk_level,annual_claims,pcp_engaged,diabetes_flag,chf_flag,copd_flag,renal_flag
35181,M35181,50,M,Central,1,0,Medium,14,0,0,0,0,0
29260,M29260,45,F,North,0,1,Medium,15,0,0,0,0,0
21439,M21439,61,M,West,1,0,Medium,14,1,0,0,0,0
17478,M17478,2,M,Central,0,0,Low,7,1,0,0,0,0
42022,M42022,32,F,East,0,1,Low,7,1,0,0,0,0


In [42]:
for idx, row in members.iterrows():
  n = int(row["chronic_count"])
  if n>0:
    chosen = np.random.choice (hcc_cols, size=n, replace= False)
    members.loc[idx,chosen]= 1

In [43]:
members.sample(5)

,member_id,age,gender,region,chronic_count,behavioral_flag,risk_level,annual_claims,pcp_engaged,diabetes_flag,chf_flag,copd_flag,renal_flag
32553,M32553,5,F,East,0,0,Low,6,1,0,0,0,0
11990,M11990,48,M,East,0,0,Low,7,0,0,0,0,0
47247,M47247,14,M,South,0,0,Low,8,1,0,0,0,0
30845,M30845,36,M,West,3,1,High,29,0,1,1,0,1
21898,M21898,66,M,Central,1,0,Medium,15,1,0,0,1,0


In [44]:
print(members[hcc_cols].mean())

diabetes_flag    0.13208
chf_flag         0.12964
copd_flag        0.13184
renal_flag       0.13374
dtype: float64


In [45]:
print((members[hcc_cols].sum(axis=1)-members["chronic_count"]).abs().sum())

0


In [46]:
print((members[hcc_cols].sum(axis=1)-members["chronic_count"]).all())

False


In [47]:
print(members.columns)

Index(['member_id', 'age', 'gender', 'region', 'chronic_count',
       'behavioral_flag', 'risk_level', 'annual_claims', 'pcp_engaged',
       'diabetes_flag', 'chf_flag', 'copd_flag', 'renal_flag'],
      dtype='object')


In [48]:
member_features = members[["member_id", "chronic_count","behavioral_flag","risk_level"]].copy()

In [49]:
member_features =  member_features.rename (columns = {"chronic_count": "num_chronic_conditions",
                                                      "behavioral_flag":"behavioral_health_flag",
                                                      "risk_level": "risk_bin"})

In [50]:
member_features= member_features.merge(
    members[["member_id"] + hcc_cols],
    on = "member_id",
    how="left"
)

In [51]:
member_features.sample(5)

,member_id,num_chronic_conditions,behavioral_health_flag,risk_bin,diabetes_flag,chf_flag,copd_flag,renal_flag
34001,M34001,0,0,Low,0,0,0,0
12521,M12521,1,0,Medium,1,0,0,0
2581,M2581,1,1,Medium,0,1,0,0
14582,M14582,1,0,Medium,0,0,1,0
43491,M43491,0,0,Low,0,0,0,0


In [52]:
members.to_parquet("members_phase1.parquet", index=False)

In [53]:
from google.colab import files
files.download("members_phase1.parquet")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [54]:
members.to_parquet("/content/drive/MyDrive/members_phase1.parquet", index=False)


In [55]:
test_members = pd.read_parquet("members_phase1.parquet")
print(test_members.columns)

Index(['member_id', 'age', 'gender', 'region', 'chronic_count',
       'behavioral_flag', 'risk_level', 'annual_claims', 'pcp_engaged',
       'diabetes_flag', 'chf_flag', 'copd_flag', 'renal_flag'],
      dtype='object')


In [56]:
print(claims.columns)

Index(['member_id', 'age', 'gender', 'region', 'chronic_count',
       'behavioral_flag', 'risk_level', 'annual_claims', 'claim_id', 'date',
       'place_of_service', 'diagnosis_group', 'admission_flag',
       'allowed_amount', 'avoidable_er', 'provider_id', 'speciality',
       'provider_region', 'pcp_engaged', 'region_multiplier'],
      dtype='object')


In [57]:
print(members.columns)

Index(['member_id', 'age', 'gender', 'region', 'chronic_count',
       'behavioral_flag', 'risk_level', 'annual_claims', 'pcp_engaged',
       'diabetes_flag', 'chf_flag', 'copd_flag', 'renal_flag'],
      dtype='object')


In [58]:
claims= claims.rename(columns={"avoidable_er":"avoidable_er_flag"})

In [59]:
p = 0.35

claims["avoidable_er_flag"] = np.where(
    (claims["place_of_service"] == "ER") &
    (claims["admission_flag"] == 0) &
    (claims["diagnosis_group"] == "low_acuity") &
    (np.random.rand(len(claims)) < p),
    1,
    0
)


In [60]:
print(claims.columns)


Index(['member_id', 'age', 'gender', 'region', 'chronic_count',
       'behavioral_flag', 'risk_level', 'annual_claims', 'claim_id', 'date',
       'place_of_service', 'diagnosis_group', 'admission_flag',
       'allowed_amount', 'avoidable_er_flag', 'provider_id', 'speciality',
       'provider_region', 'pcp_engaged', 'region_multiplier'],
      dtype='object')


In [61]:
claims["er_visit"] = (claims["place_of_service"]=="ER").astype(int)

In [62]:
claims["no_admission"]= (claims["admission_flag"]==0).astype(int)

In [63]:
claims["low_acuity"]= (claims["diagnosis_group"]=="low_acuity").astype(int)

In [64]:
print(claims.columns)

Index(['member_id', 'age', 'gender', 'region', 'chronic_count',
       'behavioral_flag', 'risk_level', 'annual_claims', 'claim_id', 'date',
       'place_of_service', 'diagnosis_group', 'admission_flag',
       'allowed_amount', 'avoidable_er_flag', 'provider_id', 'speciality',
       'provider_region', 'pcp_engaged', 'region_multiplier', 'er_visit',
       'no_admission', 'low_acuity'],
      dtype='object')


In [65]:
claims.sample(5)

,member_id,age,gender,region,chronic_count,behavioral_flag,risk_level,annual_claims,claim_id,date,...,allowed_amount,avoidable_er_flag,provider_id,speciality,provider_region,pcp_engaged,region_multiplier,er_visit,no_admission,low_acuity
318476,M30575,62,F,North,1,1,Medium,14,C318476,2024-02-17,...,98.505521,0,P981,Specialist,Costal,1,1.10,0,1,0
238108,M22779,27,F,North,1,0,Low,7,C238108,2024-05-30,...,265.268168,0,P38,Specialist,South,1,1.10,0,1,0
11145,M1066,33,F,West,2,0,Medium,12,C11145,2024-03-15,...,219.533597,0,P205,Specialist,North,1,1.05,0,1,0
490244,M46991,48,M,Central,1,0,Medium,15,C490244,2024-07-16,...,261.234429,0,P683,Specialist,North,0,0.97,0,1,1
371171,M35678,54,M,Central,0,1,Low,8,C371171,2024-11-12,...,153.279063,0,P389,Specialist,West,1,0.97,0,1,1


In [66]:
from datetime import datetime, timedelta

In [67]:
cutoff_date = claims["date"].max() - pd.Timedelta(days = 90)

In [68]:
cutoff_date

Timestamp('2024-10-01 00:00:00')

In [69]:
er_90d = (
    claims[
        (claims["er_visit"]==1) & (claims["date"]>= cutoff_date)
        ]
    .groupby("member_id")
    .size()
    .reset_index(name="prior_er_visits_90d")
)

In [70]:
member_features = member_features.merge(
    er_90d,
    on ="member_id",
    how= "left"
)

In [71]:
member_features["prior_er_visits_90d"]= member_features["prior_er_visits_90d"].fillna(0)

In [72]:
member_features.sample(5)

,member_id,num_chronic_conditions,behavioral_health_flag,risk_bin,diabetes_flag,chf_flag,copd_flag,renal_flag,prior_er_visits_90d
40262,M40262,0,0,Low,0,0,0,0,0.0
7831,M7831,0,0,Low,0,0,0,0,0.0
39209,M39209,0,0,Low,0,0,0,0,0.0
19821,M19821,0,0,Low,0,0,0,0,0.0
9916,M9916,0,0,Low,0,0,0,0,0.0


In [73]:
pcp_90d = (
    claims[
        (claims["speciality"] =="PrimaryCare") &
        (claims["date"]>= cutoff_date)
    ]
    .groupby("member_id")
    .size()
    .reset_index(name = "pcp_touch_90d")
)

In [74]:
member_features= member_features.merge(
    pcp_90d,
    on= "member_id",
    how="left"
)

In [75]:
member_features["pcp_touch_90d"]= member_features["pcp_touch_90d"].fillna(0)

In [76]:
member_features.sample(5)

,member_id,num_chronic_conditions,behavioral_health_flag,risk_bin,diabetes_flag,chf_flag,copd_flag,renal_flag,prior_er_visits_90d,pcp_touch_90d
46516,M46516,0,0,Low,0,0,0,0,0.0,0.0
24449,M24449,1,0,Low,1,0,0,0,0.0,0.0
42342,M42342,0,1,Low,0,0,0,0,0.0,1.0
7482,M7482,0,0,Low,0,0,0,0,0.0,0.0
34356,M34356,0,0,Low,0,0,0,0,0.0,0.0


In [77]:
target =  (
    claims.groupby("member_id")["avoidable_er_flag"]
    .max()
    .reset_index(name="has_avoidable_er")
)

In [78]:
member_features = member_features.drop(columns=["has_avoidable_er"], errors="ignore")


In [79]:
member_features = member_features.merge(
    target,
    on = "member_id",
    how= "left"
)

In [80]:
member_features["has_avoidable_er"] = member_features["has_avoidable_er"].fillna(0)

In [81]:
member_features.sample(5)

,member_id,num_chronic_conditions,behavioral_health_flag,risk_bin,diabetes_flag,chf_flag,copd_flag,renal_flag,prior_er_visits_90d,pcp_touch_90d,has_avoidable_er
5380,M5380,0,0,Low,0,0,0,0,0.0,0.0,0
7844,M7844,0,0,Low,0,0,0,0,1.0,1.0,0
17358,M17358,0,0,Low,0,0,0,0,1.0,1.0,0
6198,M6198,0,0,Low,0,0,0,0,0.0,1.0,0
47478,M47478,2,0,Medium,0,1,0,1,0.0,1.0,0


In [82]:
member_features = member_features[[
    "member_id",
    "num_chronic_conditions",
    "diabetes_flag",
    "chf_flag",
    "copd_flag",
    "renal_flag",
    "behavioral_health_flag",
    "prior_er_visits_90d",
    "pcp_touch_90d",
    "risk_bin",
    "has_avoidable_er"
]]


In [83]:
member_features.sample(5)

,member_id,num_chronic_conditions,diabetes_flag,chf_flag,copd_flag,renal_flag,behavioral_health_flag,prior_er_visits_90d,pcp_touch_90d,risk_bin,has_avoidable_er
10791,M10791,0,0,0,0,0,0,0.0,1.0,Low,0
25741,M25741,0,0,0,0,0,0,1.0,1.0,Low,0
39216,M39216,0,0,0,0,0,0,0.0,2.0,Low,0
3420,M3420,1,0,0,0,1,0,0.0,2.0,Low,0
40067,M40067,2,0,1,0,1,0,2.0,0.0,High,1


In [84]:
member_features.shape

(50000, 11)

In [85]:
member_features.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 11 columns):
 #   Column                  Non-Null Count  Dtype   
---  ------                  --------------  -----   
 0   member_id               50000 non-null  object  
 1   num_chronic_conditions  50000 non-null  int64   
 2   diabetes_flag           50000 non-null  int64   
 3   chf_flag                50000 non-null  int64   
 4   copd_flag               50000 non-null  int64   
 5   renal_flag              50000 non-null  int64   
 6   behavioral_health_flag  50000 non-null  int64   
 7   prior_er_visits_90d     50000 non-null  float64 
 8   pcp_touch_90d           50000 non-null  float64 
 9   risk_bin                50000 non-null  category
 10  has_avoidable_er        50000 non-null  int64   
dtypes: category(1), float64(2), int64(7), object(1)
memory usage: 3.9+ MB


In [86]:
member_features.isna().sum()

,0
member_id,0
num_chronic_conditions,0
diabetes_flag,0
chf_flag,0
copd_flag,0
renal_flag,0
behavioral_health_flag,0
prior_er_visits_90d,0
pcp_touch_90d,0
risk_bin,0


In [87]:
member_features["has_avoidable_er"].mean()

np.float64(0.25436)

In [88]:
claims["avoidable_er_flag"].mean()

np.float64(0.029385360838189652)

In [89]:
claims["er_visit"].mean()

np.float64(0.21037416988609553)

In [90]:
member_features.to_parquet("member_features.parquet", index=False)

In [91]:
from google.colab import files
files.download("member_features.parquet")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>